# Identifying AI Image — Colab workbench

Runs the evidence-fusion MVP (provenance → watermarks → forensics → verdict) in Colab.

**Setup options** (pick one in the cell below):
- **A. Zip upload** — upload `ai-image-id-mvp.zip` when prompted.
- **B. GitHub** — once the repo is pushed, replace the URL and use the pip line instead.

In [ ]:
# System dependency (exiftool) — required for metadata extraction
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!exiftool -ver

In [ ]:
# --- Option A: upload the zip ---
from google.colab import files
uploaded = files.upload()  # select ai-image-id-mvp.zip
!unzip -oq ai-image-id-mvp.zip
%pip install -q -e ./ai-image-id-mvp

# --- Option B: from GitHub (uncomment once pushed) ---
# %pip install -q git+https://github.com/<you>/ai-image-id.git

In [ ]:
# Sanity check
from ai_image_id.main import analyze_image
print("package OK")

## Analyze any image

Upload an image and run the full pipeline. Try a Firefly/DALL·E export (C2PA), a
Meta AI image (IPTC tag), a raw SD/SDXL output (invisible watermark), and a phone photo.

In [ ]:
from google.colab import files
import json

up = files.upload()
path = next(iter(up))
result = analyze_image(path)
print(json.dumps(json.loads(result.model_dump_json()), indent=2))

## Demo: watermark round-trip

Embeds the SDXL 48-bit payload into a synthetic image, then verifies the pipeline
flags it as `verified`. Also shows fragility: JPEG-recompress the marked image and
watch bit accuracy drop — absence of a watermark is non-evidence.

In [ ]:
import numpy as np
from PIL import Image
from ai_image_id.watermark import dwt_dct, SDXL_BITS

rng = np.random.default_rng(0)
y, x = np.mgrid[0:512, 0:512]
base = np.stack([120+60*np.sin(x/97), 100+50*np.cos(y/83), 90+40*np.sin((x+y)/71)], axis=-1)
clean = np.clip(base + rng.normal(0, 6, base.shape), 0, 255).astype(np.uint8)

marked = dwt_dct.embed(clean, SDXL_BITS)
Image.fromarray(marked).save("marked.png")            # lossless
Image.fromarray(marked).save("marked_q70.jpg", quality=70)  # lossy attack

for f in ["marked.png", "marked_q70.jpg"]:
    r = analyze_image(f)
    wm = next(w for w in r.evidence.watermarks if w.scheme == "dwtDct")
    print(f"{f}: verdict={r.ai_verdict.value}, bit_accuracy={wm.bit_accuracy}")

## Phase 2 workspace (learned detector)

Next steps, per the implementation plan — switch runtime to **GPU** for step 2:
1. Data prep: GenImage subset with matched JPEG-Q and resolution distributions
2. Precompute frozen DINOv2/CLIP embeddings (one pass, GPU)
3. Train attention-pooling head / linear probe (minutes, CPU-fine)
4. Temperature-scale calibration + cross-generator eval table
5. Export ONNX, add `detector` evidence to the fusion engine

In [ ]:
# Phase 2 scaffold lands here — e.g.:
# %pip install -q torch timm
# import torch; torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')